# Análisis de la base de datos 
## Universidad de los Andes - Smurfit Westrock
### Poyecto Intermedio de Consultoría Empresarial (PICE) 202520
Daniel Benavides

This code performs an exploratory and preparatory analysis of Smurfit Westrock’s payment data. It begins by importing and cleaning raw datasets from Excel or CSV files, addressing missing values, duplicates, and inconsistencies. The data is then transformed through normalization of numerical variables and encoding of categorical ones such as suppliers, cost centers, and expense types. Exploratory Data Analysis (EDA) is conducted to visualize payment distributions, identify outliers and temporal trends, and examine correlations among key variables. Additionally, feature engineering is applied to create new indicators that capture behavioral patterns and transaction frequency, ensuring the dataset is ready for anomaly detection models. This analysis provides preliminary insights and recommendations to guide the development of Machine Learning models and improve overall data quality.

In [ ]:
# Data extraction libraries
import numpy as np
import pandas as pd

# Data visualizaton libraries 
import matplotlib.pyplot as plt
import seaborn as sns
import bokeh
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "browser"
import altair as alt
from prettytable import PrettyTable

from matplotlib import font_manager
plt.rcParams['font.family'] = 'Arial'

Data downloaded as Excel files

In [ ]:
from Excel_reader import read_excel_files

# df_excel = read_excel_files("PICE BD 2025-Parte 1.xlsx", 
                            # "PICE BD 2025-Parte 2.xlsx", 
                            # "PICE BD 2025-Parte 3.xlsx")

### PICE BD 2025 JOINT CSV FILE

Data downloaded as CSV file (ideal)

In [ ]:
# df_csv = df_excel.to_csv("PICE BD 2025.csv", index=False)
df_csv = pd.read_csv("PICE BD 2025.csv", low_memory=False)
df_csv.info()

df_csv.head()

# 1. Data Cleaning and Transformation

In [ ]:
from sklearn.preprocessing import StandardScaler, RobustScaler

db = df_csv.copy()
db.rename(columns={"En moneda de la sociedad": "Monto",
                   "Se ha anulado el Documento": "Estatus de anulación"}, inplace=True)

# En el dataframe, existen registro anulados marcados ("X") y registros no anulados (NaN)
# Transformar "X" a 1 y NaN a 0 en la columna "Estatus de anulación"
db["Estatus de anulación"] = db["Estatus de anulación"].apply(lambda x: 1 if x == "X" else 0)

# Transform date columns
db["Fecha Contable"] = pd.to_datetime(db["Fecha Contable"], errors='coerce')
db["Año"] = db["Fecha Contable"].dt.year.round(0).astype('Int64')
db["Mes"] = db["Fecha Contable"].dt.month.round(0).astype('Int64')
db["Dia"] = db["Fecha Contable"].dt.day.round(0).astype('Int64')

db.head()

### 1.1 Scaling of numeric variables

In [ ]:
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()
db['Monto_Normalized'] = scaler.fit_transform(db[['Monto']])
db['Cantidad_Normalized'] = scaler.fit_transform(db[['Cantidad']])

db.head()

### Relevant variables

In [ ]:
variables = ["Denominación","Centro de Coste", "Usuario", "Mes", 
             "Clase", "Clase de Movimiento V", "Tipo de Documento", "Centro de Beneficio", 
             "Clase de Factura", "Estatus de anulación",
             "Sector"]

db[variables].nunique().sort_values(ascending=False)

table = PrettyTable()
table.field_names = ["Variable", "Unique Values"]
for var, unique_count in db[variables].nunique().sort_values(ascending=False).items():
    table.add_row([var, unique_count])
print(table)

In [ ]:
# Count percentage of missing values per column
missing_percentage = db.isnull().mean() * 100
missing_percentage = missing_percentage[missing_percentage > 0].sort_values(ascending=False)
round(missing_percentage, 2)

table = PrettyTable()
table.field_names = ["Variable", "Missing Percentage"]
for var, perc in missing_percentage.items():
    table.add_row([var, f"{perc:.2f}%"])
print(table)


### Variables to impute (parameter for 2. and 3.)

In [ ]:
variables_to_impute = [ "Sector", "Clase de Factura", "Clase de Movimiento V",
                        "Centro de Coste", "Ledger", "Cantidad",
                        "Centro", "Hora", "Clase" ]

# 2. Missing Data Analysis
The missing data analysis corresponds to the reuslts provided in the python file **Missing_Data_Analysis**

In [ ]:
from Missing_Data_Analysis import comprehensive_missingness_analysis

results = comprehensive_missingness_analysis(db, variables_to_impute)

# 3. MICE Imputation
MICE (Multiple Imputation by Chained Equations) imputation is a method for handling missing data in a dataset by iteratively using predictive models to fill in the blanks. It creates multiple completed datasets, which are then analyzed separately, and the results are pooled to get a single, more robust estimate. 

In [ ]:
from MICE_Imputation import mice_imputation

db_imputed = mice_imputation(db, variables_to_impute)

# 4. Analysis of Cleaned and Imputed Data

In [ ]:
db_imputed.head()

In [ ]:
db_imputed.info()

In [ ]:
# Retrieve top 5 most transacting users
cat_var = ["Denominación","Centro de Coste", "Usuario", 
            "Clase", "Clase de Movimiento V", "Centro de Beneficio", 
            "Clase de Factura", "Sector"]

for var in cat_var:
    print(f"Top 5 for {var}:")
    print(db_imputed[var].value_counts().head(5))
    print("\n")

In [ ]:
# Binary variables
bin_var = ["Estatus de anulación", "Tipo de Documento"]

for var in bin_var:
    print(f"Value counts for {var}:")
    print(db_imputed[var].value_counts())
    print("\n")

### 4.1 Cantidad de transacciones por **día y mes**

In [ ]:
# Heatmap of number of transactions by day and month
db_heatmap = db_imputed.pivot_table(index="Mes", columns="Dia", values="Monto", aggfunc="count", fill_value=0)
db_heatmap.index = db_heatmap.index.astype(int)
db_heatmap.columns = db_heatmap.columns.astype(int)

plt.figure(figsize=(20, 10))
sns.heatmap(db_heatmap, cmap="YlGnBu", annot=True, fmt="d")
plt.title("Transacciones diarias")

### 4.2 Montos transados por **Centro de Coste**

In [ ]:
pio.templates["plotly"].layout.font.family = "Arial"
pio.templates["plotly_white"].layout.font.family = "Arial"
pio.templates.default = "plotly"

In [ ]:
# Monto transado por Centro de Coste
db_cc_ano = db_imputed.groupby(["Centro de Coste"])["Monto"].sum().reset_index()

px.bar(db_cc_ano,
       x="Centro de Coste",
       y="Monto",
       title="Monto transado por Centro de Coste")

In [ ]:
# Top 5 Centros de Coste por monto transado
top_5_cc = db_cc_ano.sort_values(by="Monto", ascending=False).head(5)
top_5_cc

In [ ]:
# Bottom 5 Centros de Coste por monto transado
bottom_5_cc = db_cc_ano.sort_values(by="Monto", ascending=True).head(5)
bottom_5_cc

In [ ]:
# Promedio Monto transado por Centro de Coste
db_cc_ano = db_imputed.groupby(["Centro de Coste"])["Monto"].mean().reset_index()

px.bar(db_cc_ano,
       x="Centro de Coste",
       y="Monto",
       title="Promedio del Monto transado por Centro de Coste")

### 4.3 Cantidad de transacciones por **hora y día**

In [ ]:
# Transacciones by hour and day
alt.data_transformers.enable('vegafusion')

db_imputed['Fecha'] = pd.to_datetime(db_imputed[['Año', 'Mes', 'Dia']].rename(columns={
    'Año': 'year',
    'Mes': 'month', 
    'Dia': 'day'
}))

# Extract day of week name (Monday, Tuesday, ...)
db_imputed['Dia_Semana'] = db_imputed['Fecha'].dt.day_name()

# Extract hour from Hora column
db_imputed['Hora'] = pd.to_datetime(db_imputed['Hora'], format='%H:%M:%S', errors='coerce').dt.hour

print(db_imputed[['Año', 'Mes', 'Dia', 'Fecha', 'Dia_Semana']].head())

In [ ]:
# Count transactions by day of week and hour
chart_data = db_imputed.groupby(['Dia_Semana', 'Hora']).size().reset_index(name='count')

# Create the chart
alt.Chart(chart_data).mark_circle().encode(
    x=alt.X('Hora:O', title='Hora del día'),
    y=alt.Y('Dia_Semana:O', title='Día de la semana', 
            sort=['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']),
    size=alt.Size('count:Q', title='Cantidad de transacciones', scale=alt.Scale(range=[0, 1000])),
    tooltip=['Dia_Semana', 'Hora', 'count']
).properties(
    width=500,
    height=300,
    title='Cantidad de transacciones por día de la semana y hora del día'
)

### 4.4 Cantidad de transacciones 